# Phase 6 — AD-Guided Adaptive Sampling

Use the applicability domain to choose **where to run more SWMM**: add points in the sparse, high-error regions Phase 5 flagged, retrain, and check the surrogate improves there. Controlled comparison: **AD-guided (maximin) vs random**, same budget, judged on a fixed held-out SWMM test set.



## 1. Setup: training set, surrogate, fixed eval set, SWMM helpers

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Open this notebook from the project folder or the notebooks folder.
PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
assert (PROJECT / "notebooks").is_dir(), "Open from the project or notebooks folder."
DATA_DIR = PROJECT / "output"
RUNS_DIR = PROJECT / "runs"
RUNS_DIR.mkdir(exist_ok=True)
SWMM_EXE = Path(r"D:\EPA SWMM 5.2.4 (64-bit)\runswmm.exe")
BASE_INP = PROJECT / "notebooks" / "base_model.inp"
SEED = 42

sys.path.insert(0, str(PROJECT / "notebooks"))
from swmm_helpers import write_scenario, run_swmm, parse_rpt

FEATURES = ["rain_mult", "imperv_mult", "surf_n", "infil_min", "pipe_n"]

df = pd.read_csv(DATA_DIR / "dataset.csv")
df = df[df["status"] == "ok"].reset_index(drop=True)
Xtrain = df[FEATURES].values
ytrain = df["peak_discharge_cfs"].values

# fixed held-out evaluation set = Phase-5 probes (already have SWMM ground truth)
probes = pd.read_csv(DATA_DIR / "ad_probes.csv")
Xeval = probes[FEATURES].values
yeval = probes["swmm_peak"].values

print(f"training: {len(df)} | fixed eval (Phase-5 probes): {len(probes)}")
assert SWMM_EXE.exists()

## 2. Candidate pool (wide box)

3,000 fresh proposals over the box 50% wider than training each side (physically clipped), a different seed from the probes. No SWMM yet.

In [ ]:
train_mean, train_std = Xtrain.mean(0), Xtrain.std(0)
def zscore(X):
    return (np.asarray(X) - train_mean) / train_std

lo, hi = Xtrain.min(0), Xtrain.max(0)
center, half = (lo + hi) / 2, (hi - lo) / 2
WIDEN, N_POOL = 1.5, 3000
wlo, whi = center - WIDEN * half, center + WIDEN * half

rng = np.random.default_rng(SEED + 7)
pool = rng.uniform(wlo, whi, size=(N_POOL, len(FEATURES)))
# Keep each parameter within its physical limits.
physical_limits = {
    "rain_mult": (0.1, None),
    "imperv_mult": (0.1, None),
    "surf_n": (0.005, None),
    "infil_min": (0.01, 4.0),  # Below the Horton maximum rate.
    "pipe_n": (0.005, None),
}
for column, feature in enumerate(FEATURES):
    lower, upper = physical_limits[feature]
    pool[:, column] = np.clip(pool[:, column], lower, upper)
print("candidate pool:", pool.shape)

## 3. Select 150 points: adaptive (maximin) vs random

**Adaptive:** greedily take the candidate farthest (standardized Euclidean) from the current training set, updating the running min-distance after each pick so chosen points spread out. **Random:** 150 drawn uniformly from the pool.

In [ ]:
from scipy.spatial.distance import cdist

N_ADD = 150
Ztrain, Zpool = zscore(Xtrain), zscore(pool)

# --- adaptive: greedy maximin ---
nearest_distance = cdist(Zpool, Ztrain).min(axis=1)   # dist from each pool pt to nearest training pt
selected = []
for _ in range(N_ADD):
    candidate = nearest_distance.argmax()
    selected.append(candidate)
    distances = cdist(Zpool, Zpool[candidate:candidate + 1]).ravel()
    nearest_distance = np.minimum(nearest_distance, distances)
    nearest_distance[candidate] = -np.inf  # Do not select the same point again.
adaptive_idx = np.array(selected)

# --- random baseline ---
random_idx = np.random.default_rng(SEED + 99).choice(N_POOL, size=N_ADD, replace=False)

print(f"selected {N_ADD} adaptive + {N_ADD} random from pool")

## 4. Run SWMM on the selected points

In [ ]:
base_text = BASE_INP.read_text()

def swmm_peak_for(points, tag):
    peaks = []
    for i, p in enumerate(points):
        inp = RUNS_DIR / f"{tag}_{i:03d}.inp"
        inp.write_text(write_scenario(base_text, dict(zip(FEATURES, p))))
        rpt, out, proc = run_swmm(inp, SWMM_EXE)
        peak = np.nan
        if proc.returncode == 0 and rpt.exists():
            try:
                peak = parse_rpt(rpt.read_text())["peak_discharge_cfs"]
            except (ValueError, IndexError) as error:
                print(f"{tag} run {i} has an incomplete report: {error}")
        peaks.append(peak)
        if out.exists():
            out.unlink()
    return np.array(peaks)

adaptive_samples = pool[adaptive_idx]
adaptive_peaks = swmm_peak_for(adaptive_samples, "adapt")
random_samples = pool[random_idx]
random_peaks = swmm_peak_for(random_samples, "rand")
print(f"SWMM ground truth obtained  (adaptive {np.isfinite(adaptive_peaks).sum()}, random {np.isfinite(random_peaks).sum()})")

## 5. Retrain 3 surrogates & evaluate on the held-out set

Baseline (500) vs 500+random vs 500+adaptive, all judged on the fixed Phase-5 probes.

In [ ]:
def make_model():
    return XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                        subsample=0.9, random_state=SEED)

valid_adaptive, valid_random = np.isfinite(adaptive_peaks), np.isfinite(random_peaks)
sets = {
    "baseline_500":        (Xtrain, ytrain),
    f"+{int(valid_random.sum())}_random":   (np.vstack([Xtrain, random_samples[valid_random]]), np.concatenate([ytrain, random_peaks[valid_random]])),
    f"+{int(valid_adaptive.sum())}_adaptive": (np.vstack([Xtrain, adaptive_samples[valid_adaptive]]), np.concatenate([ytrain, adaptive_peaks[valid_adaptive]])),
}

rows, preds = [], {}
for label, (Xtr, ytr) in sets.items():
    model = make_model()
    model.fit(Xtr, ytr)
    pred = model.predict(Xeval)
    preds[label] = pred
    rows.append({"model": label, "n_train": len(Xtr),
                 "MAE":  mean_absolute_error(yeval, pred),
                 "RMSE": mean_squared_error(yeval, pred) ** 0.5,
                 "R2":   r2_score(yeval, pred)})

res = pd.DataFrame(rows)
print(res.round(3).to_string(index=False))

## 6. Where did adaptive help? Error by distance quintile

Split the eval set into Mahalanobis-distance quintiles (5 = farthest from training) and compare mean |error| across the three surrogates.

In [ ]:
covariance = np.cov(Xtrain, rowvar=False)
inverse_covariance = np.linalg.inv(covariance)
def maha(X):
    differences = np.asarray(X) - train_mean
    return np.sqrt(((differences @ inverse_covariance) * differences).sum(axis=1))

d_eval = maha(Xeval)
base_lab = "baseline_500"
rand_lab = f"+{int(valid_random.sum())}_random"
adap_lab = f"+{int(valid_adaptive.sum())}_adaptive"

err = pd.DataFrame({
    "quintile": pd.qcut(d_eval, 5, labels=[1, 2, 3, 4, 5]),
    "baseline": np.abs(yeval - preds[base_lab]),
    "random":   np.abs(yeval - preds[rand_lab]),
    "adaptive": np.abs(yeval - preds[adap_lab]),
})
by_q = err.groupby("quintile", observed=True)[["baseline", "random", "adaptive"]].mean()
print("Mean |error| (CFS) by Mahalanobis quintile:")
print(by_q.round(3))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
by_q.plot.bar(ax=ax[0])
ax[0].set_title("Mean |error| by distance quintile")
ax[0].set_xlabel("Mahalanobis quintile (5=farthest)")
ax[0].set_ylabel("|error| (CFS)")
ax[0].tick_params(axis="x", rotation=0)
res.set_index("model")["MAE"].plot.bar(ax=ax[1], color=["#888", "#3b7dd8", "#2ca25f"])
ax[1].set_title("Overall eval MAE")
ax[1].set_ylabel("MAE (CFS)")
ax[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

## 7. Verdict + save

In [ ]:
res.to_csv(DATA_DIR / "adaptive_results.csv", index=False)

base_mae = res.loc[res.model == base_lab, "MAE"].iloc[0]
rand_mae = res.loc[res.model == rand_lab, "MAE"].iloc[0]
adap_mae = res.loc[res.model == adap_lab, "MAE"].iloc[0]

print(f"Held-out MAE:  baseline {base_mae:.3f}  ->  random {rand_mae:.3f}  ->  adaptive {adap_mae:.3f} CFS\n")
print(f"Adaptive vs baseline : {(1 - adap_mae/base_mae)*100:5.1f}% error reduction")
print(f"Adaptive vs random   : {(1 - adap_mae/rand_mae)*100:5.1f}% better than same-budget random")
print("\nSaved -> output/adaptive_results.csv")